In [1]:
import pandas as pd

users = pd.read_csv("../data/raw/dim_users.csv")
transactions = pd.read_csv("../data/raw/fact_transactions.csv")

In [2]:
# User-Level Features
user_features = (
    transactions
    .groupby("user_id")
    .agg(
        transaction_count=("transaction_id","count"),
        total_transaction_amount=("amount","sum"),
        avg_transaction_amount=("amount","mean"),
        last_transaction_date=("transaction_date","max")
    )
    .reset_index()
)

In [3]:
# Churn Flag
user_features["last_transaction_date"] = pd.to_datetime(
    user_features["last_transaction_date"]
)

user_features["churn_flag"] = (
    user_features["last_transaction_date"] < "2025-10-01"
).astype(int)

In [ ]:
# Join with Users
df = users.merge(
    user_features,
    on="user_id",
    how="inner"
)

In [5]:
# Check Churn Distribution
df["churn_flag"].value_counts()

churn_flag
0    17814
1     7186
Name: count, dtype: int64

In [ ]:
# Check Shape
print(df.shape)

(25000, 12)


In [7]:
# Columns
df.columns.tolist()

['user_id',
 'signup_date',
 'city',
 'age',
 'acquisition_channel',
 'device_type',
 'kyc_status',
 'transaction_count',
 'total_transaction_amount',
 'avg_transaction_amount',
 'last_transaction_date',
 'churn_flag']

In [8]:
# Feature Selection and Data Preparation for Churn Prediction Model
features = [
    "age",
    "city",
    "acquisition_channel",
    "device_type",
    "kyc_status",
    "transaction_count",
    "total_transaction_amount",
    "avg_transaction_amount"
]

target = "churn_flag"

df_ml = pd.get_dummies(
    df[features],
    drop_first=True
)

df_ml[target] = df[target]

print(df_ml.shape)

(25000, 16)


In [9]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X = df_ml.drop("churn_flag", axis=1)
y = df_ml["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (20000, 15)
Testing Shape: (5000, 15)


In [10]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [ ]:
# Model Predictions
predictions_rd = model.predict(X_test)

predictions_rd[:10]

array([0, 0, 1, 0, 0, 0, 0, 1, 0, 0])

In [13]:
# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, predictions_rd)

print("Accuracy:", round(accuracy,4))

print("\nClassification Report:")
print(classification_report(y_test, predictions_rd))

Accuracy: 0.6884

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.89      0.80      3563
           1       0.41      0.19      0.26      1437

    accuracy                           0.69      5000
   macro avg       0.57      0.54      0.53      5000
weighted avg       0.64      0.69      0.65      5000



- Developed a Random Forest churn prediction model achieving 68.8% accuracy.

- Identified user behavior and transaction activity as key predictors of customer churn.

- Demonstrated end-to-end machine learning workflow including feature engineering, model training, and evaluation.


In [14]:
# Feature Importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    by="importance",
    ascending=False
)

importance.head(10)

,feature,importance
2,total_transaction_amount,0.278048
3,avg_transaction_amount,0.257265
0,age,0.203971
1,transaction_count,0.109918
13,device_type_iOS,0.024087
14,kyc_status_Pending,0.013622
10,acquisition_channel_Meta,0.013587
9,acquisition_channel_Google,0.013205
11,acquisition_channel_Organic,0.012937
12,acquisition_channel_Referral,0.012885
